# Task 03 — Gold: Daily Revenue Aggregation

**Workshop: Final Pipeline | Layer 3 of 3**

## Goal

Read `silver.lab_orders`, aggregate to daily metrics, write to `gold.lab_daily_orders`.

```
silver.lab_orders
        |
        v  to_date * groupBy * agg
        v
gold.lab_daily_orders
```

## Expected output schema

| Column | Type | Description |
|--------|------|-------------|
| order_date | DateType | Calendar day from `order_datetime` |
| order_count | LongType | Total orders that day |
| total_revenue | DoubleType | SUM of `net_amount` |
| avg_order_value | DoubleType | AVG of `net_amount`, rounded to 2 dp |

> Use `net_amount` (after discount), not `total_amount`. One row = one calendar day.


In [ ]:
%run ../../setup/00_setup

In [ ]:
dbutils.widgets.text("catalog",       CATALOG,       "Catalog")
dbutils.widgets.text("silver_schema", SILVER_SCHEMA, "Silver Schema")
dbutils.widgets.text("gold_schema",   GOLD_SCHEMA,   "Gold Schema")

catalog       = dbutils.widgets.get("catalog")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema   = dbutils.widgets.get("gold_schema")

source_table = f"{catalog}.{silver_schema}.lab_orders"
target_table = f"{catalog}.{gold_schema}.lab_daily_orders"

print(f"Source : {source_table}")
print(f"Target : {target_table}")

---

## Exercise 1 — Imports

Import aggregate functions and `to_date` from `pyspark.sql.functions`.

> Note: Python has built-in `sum()`, `round()`, and `min()`.
> Importing the same names from `pyspark.sql.functions` shadows them here — that is intentional.


**💡 Guidance — Exercise 1**

Import the aggregate functions and date utility needed for the Gold aggregation.

**Important note on name shadowing**
Python has built-in functions named `sum()`, `round()`, and `min()`. Importing the same names from `pyspark.sql.functions` **shadows** those built-ins in this namespace — this is intentional. The PySpark versions operate on `Column` expressions, not Python scalars.

**Key imports**
- `to_date(col_expr)` — strips the time component from a `TimestampType` column, returning a `DateType` (e.g. `2024-03-15T14:23:00` → `2024-03-15`)
- `count("*")` — counts all rows in a group, including those with nulls in other columns
- `sum("col")` — sums non-null values in the column within each group
- `avg("col")` — mean of non-null values; wrap with `round(..., 2)` inside `.agg()` to limit decimal places
- `round(col_expr, scale)` — the first argument must be a `Column` expression (e.g. the result of `avg(...)`)

**Things to think about**
- What is the difference between `count("*")` and `count("order_id")` in this dataset?
- Why must you write `round(avg("net_amount"), 2)` rather than `avg("net_amount").round(2)`?

In [ ]:
# YOUR CODE HERE

---

## Exercise 2 — Read the Silver table


**💡 Guidance — Exercise 2**

Read the Silver `lab_orders` table before aggregating it.

Task 02 Notebook (Silver) must have completed successfully before this cell runs. Use `spark.table(source_table)` — the variable is already set in the widget cell above. Store the result as `orders_silver`.

At this point the Silver table should have `order_datetime` as a `TimestampType` and `net_amount` as a `DoubleType` — both columns are needed for the Gold aggregation.

**Things to think about**
- Why does this notebook read from the Silver table rather than re-applying Silver transformations itself?
- What would happen to the Gold aggregation if `net_amount` contained `null` values?

In [ ]:
# Read the Silver table into orders_silver (Task 02 must have run first)
orders_silver = # YOUR CODE HERE

In [ ]:
print(f"Silver rows : {orders_silver.count():,}")
orders_silver.printSchema()

---

## Exercise 3 — Extract `order_date` from `order_datetime`

Add a new column `order_date` (DateType) by extracting the calendar date
from `order_datetime` (TimestampType).


**💡 Guidance — Exercise 3**

Extract the calendar date from the `order_datetime` timestamp column.

**to_date() function**
`to_date(col("order_datetime"))` removes the time component and returns a `DateType` value (e.g. `2024-03-15T14:23:00` → `2024-03-15`). This is the column you will group by in Exercise 4 to produce one row per calendar day.

Add it as a new column using `.withColumn("order_date", to_date(col("order_datetime")))` on `orders_silver`. Store the result (or proceed to use it directly inline in the `groupBy()` in Exercise 4 if you prefer a single chain).

**Things to think about**
- What would happen to the aggregation row count if you grouped by `order_datetime` (timestamp, includes time) instead of `order_date` (date only)?

In [ ]:
# Add order_date (DateType) by extracting the date from order_datetime (TimestampType)
# Use to_date(col("order_datetime")) with withColumn
orders_with_date = (
    orders_silver
    # YOUR CODE HERE
)

---

## Exercise 4 — Aggregate: one row per day

Group by `order_date`, compute the four metrics, sort ascending by date.
Name the result `daily_df`.


**💡 Guidance — Exercise 4**

Produce one summary row per calendar day by aggregating over `order_date`.

**groupBy().agg() pattern**
Chain `groupBy("order_date")` → `.agg(...)` with four expressions inside a single call:
1. `count("*").alias("order_count")` — total orders on that day
2. `sum("net_amount").alias("total_revenue")` — cumulative net revenue
3. `round(avg("net_amount"), 2).alias("avg_order_value")` — daily average order size

Use `net_amount` (after discount), **not** `total_amount`. Always name result columns with `.alias(...)` — without it Spark uses internal names like `sum(net_amount)`.

**Ordering the result**
Chain `.orderBy("order_date")` to return the daily rows in chronological order. Store the final DataFrame as `daily_df`.

**Things to think about**
- Why does `round(avg("net_amount"), 2)` work? What type does `avg("net_amount")` return?
- What would the Gold table look like if the Silver data contained duplicate `order_id` rows?

In [ ]:
# Group by order_date, compute:
#   order_count    = count("*")
#   total_revenue  = sum("net_amount")
#   avg_order_value = round(avg("net_amount"), 2)
# Order the result by order_date ascending
daily_df = (
    orders_with_date
    # YOUR CODE HERE
)

In [ ]:
print(f"Daily rows : {daily_df.count():,}  (= distinct order dates)")
daily_df.show(15, truncate=False)

---

## Exercise 5 — Write to Gold as a managed Delta table


**💡 Guidance — Exercise 5**

Write `daily_df` to a managed Delta table as the Gold layer of the pipeline.

**Difference from Bronze/Silver writes**
Gold is always fully rebuilt from Silver on every run. Use `mode("overwrite")`. Notice that unlike Silver, you do **not** need `option("overwriteSchema", "true")` — the Gold schema is fixed and does not change between runs.

**Pipeline completion**
After the write, this notebook calls `dbutils.notebook.exit()` with a JSON payload. When the Databricks Workflow receives `"status": "SUCCESS"` from all three notebooks, it marks the job run as `Succeeded`. If any notebook fails or exits with an error status, the downstream tasks are not started — and you can use **Repair Run** to rerun only the failed task.

**Things to think about**
- Why can Gold be freely rebuilt from Silver, but Silver cannot always be rebuilt from Bronze without the original source files?
- What would be the first symptom that the Gold aggregation formula for `net_amount` was incorrect?

In [ ]:
# Write daily_df to target_table as a managed Delta table
# mode: overwrite (Gold is always fully rebuilt from Silver)
# No overwriteSchema needed — Gold schema is stable
# YOUR CODE HERE

In [ ]:
row_count = spark.table(target_table).count()
print(f"OK  {target_table}  ->  {row_count:,} daily records")
display(spark.table(target_table))

In [ ]:
import json
dbutils.notebook.exit(json.dumps({"status":"SUCCESS","table":target_table,"rows":row_count}))